In [49]:
import pandas as pd; 
df = pd.read_csv("../data/T_ONTIME_REPORTING.csv");
df.shape
import numpy as np;
from sklearn.metrics import accuracy_score, recall_score, precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression




In [50]:
df['FL_DATE'].head()

0    6/1/2026 12:00:00 AM
1    6/1/2026 12:00:00 AM
2    6/1/2026 12:00:00 AM
3    6/1/2026 12:00:00 AM
4    6/1/2026 12:00:00 AM
Name: FL_DATE, dtype: str

In [51]:
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], format='%m/%d/%Y %I:%M:%S %p')

In [52]:
df.dtypes

FL_DATE              datetime64[us]
OP_UNIQUE_CARRIER               str
ORIGIN                          str
DEST                            str
CRS_DEP_TIME                  int64
DEP_DELAY                   float64
ARR_DELAY                   float64
ARR_DEL15                   float64
CANCELLED                   float64
DIVERTED                    float64
DISTANCE                    float64
dtype: object

In [53]:
df.columns

Index(['FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN', 'DEST', 'CRS_DEP_TIME',
       'DEP_DELAY', 'ARR_DELAY', 'ARR_DEL15', 'CANCELLED', 'DIVERTED',
       'DISTANCE'],
      dtype='str')

In [54]:
df['ARR_DELAY'].head()


0   -34.0
1    13.0
2   -12.0
3    42.0
4   -23.0
Name: ARR_DELAY, dtype: float64

In [55]:
df.head()

,FL_DATE,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,DEP_DELAY,ARR_DELAY,ARR_DEL15,CANCELLED,DIVERTED,DISTANCE
0,2026-06-01,AA,ABQ,DFW,520,-8.0,-34.0,0.0,0.0,0.0,569.0
1,2026-06-01,AA,ABQ,DFW,701,4.0,13.0,0.0,0.0,0.0,569.0
2,2026-06-01,AA,ABQ,DFW,820,-11.0,-12.0,0.0,0.0,0.0,569.0
3,2026-06-01,AA,ABQ,DFW,1043,61.0,42.0,1.0,0.0,0.0,569.0
4,2026-06-01,AA,ABQ,DFW,1607,-6.0,-23.0,0.0,0.0,0.0,569.0


KNOWN before takeoff (safe inputs): FL_DATE, OP_UNIQUE_CARRIER, ORIGIN, DEST, CRS_DEP_TIME, DISTANCE
**DEP_DELAY is how late the plane actually left, which isn’t known when the prediction is made; a model using it would look great and be useless.**
Only known after (leakage, excluded): DEP_DELAY, ARR_DELAY, CANCELLED, DIVERTED
Target: ARR_DEL15

In [56]:
df['ARR_DEL15'].value_counts()

ARR_DEL15
0.0    442158
1.0    153037
Name: count, dtype: int64

In [57]:
df['CANCELLED'].value_counts()

CANCELLED
0.0    597558
1.0     10019
Name: count, dtype: int64

In [58]:
df.isnull().sum()

FL_DATE                  0
OP_UNIQUE_CARRIER        0
ORIGIN                   0
DEST                     0
CRS_DEP_TIME             0
DEP_DELAY             9554
ARR_DELAY            12382
ARR_DEL15            12382
CANCELLED                0
DIVERTED                 0
DISTANCE                 0
dtype: int64

The 12,382 blank arrival delays equal the 10,019 cancelled plus 2,363 diverted flights.

In [59]:
df['DIVERTED'].value_counts()

DIVERTED
0.0    605214
1.0      2363
Name: count, dtype: int64

In [60]:
df_clean = df.dropna(subset=['ARR_DELAY'])
df_clean.shape

(595195, 11)

In [61]:
df.dtypes

FL_DATE              datetime64[us]
OP_UNIQUE_CARRIER               str
ORIGIN                          str
DEST                            str
CRS_DEP_TIME                  int64
DEP_DELAY                   float64
ARR_DELAY                   float64
ARR_DEL15                   float64
CANCELLED                   float64
DIVERTED                    float64
DISTANCE                    float64
dtype: object

In [62]:

df_clean['ARR_DEL15'].value_counts(normalize=True)


ARR_DEL15
0.0    0.742879
1.0    0.257121
Name: proportion, dtype: float64

About 25.7% of flights arrived 15 or more minutes late.

In [63]:
df_clean['day_of_week'] = df_clean['FL_DATE'].dt.dayofweek

In [64]:
df_clean['day_of_week'].value_counts().sort_index()

day_of_week
0    103150
1     95908
2     77371
3     82193
4     81566
5     73906
6     81101
Name: count, dtype: int64

In [65]:
df_clean['dep_hour'] = df_clean['CRS_DEP_TIME'].floordiv(100)

In [66]:
df_clean['dep_hour'].min()

np.int64(0)

In [67]:
df_clean['dep_hour'].max()

np.int64(23)

In [68]:
df_clean['dep_hour'].value_counts().sort_index()

dep_hour
0       950
1       286
2        88
3        60
4        20
5     17674
6     41322
7     44971
8     39409
9     33083
10    37515
11    36162
12    34490
13    35065
14    32339
15    33996
16    32979
17    37445
18    34886
19    31912
20    28111
21    21114
22    15271
23     6047
Name: count, dtype: int64

In [69]:
df_clean.groupby('dep_hour')['ARR_DEL15'].mean()

dep_hour
0     0.196842
1     0.104895
2     0.068182
3     0.233333
4     0.300000
5     0.083965
6     0.091356
7     0.126993
8     0.139664
9     0.165704
10    0.190457
11    0.214866
12    0.249116
13    0.270355
14    0.314852
15    0.328039
16    0.361291
17    0.362825
18    0.384452
19    0.390668
20    0.387820
21    0.363882
22    0.339991
23    0.224905
Name: ARR_DEL15, dtype: float64

In [70]:
df_clean.groupby('day_of_week')['ARR_DEL15'].mean()

day_of_week
0    0.256791
1    0.197408
2    0.230293
3    0.280413
4    0.279344
5    0.249912
6    0.314361
Name: ARR_DEL15, dtype: float64

Sunday flights were late most often (31.4% arrived 15 or more minutes late), and Tuesday flights were late least often (19.7%), compared with 25.7% overall.

In [71]:
df_clean.groupby('OP_UNIQUE_CARRIER')['ARR_DEL15'].mean()

OP_UNIQUE_CARRIER
AA    0.300835
AS    0.193539
B6    0.249424
DL    0.177180
F9    0.286840
G4    0.312087
MQ    0.212598
OH    0.276078
OO    0.223860
UA    0.252953
WN    0.340212
YX    0.186237
Name: ARR_DEL15, dtype: float64

In [72]:
X = df_clean[['day_of_week', 'dep_hour', 'DISTANCE']]

In [73]:
df_clean['dep_hour'].max()

np.int64(23)

In [74]:
airlines = pd.get_dummies(df_clean['OP_UNIQUE_CARRIER'], prefix='carrier', dtype=int)

In [75]:
X = pd.concat([X, airlines], axis=1)

In [76]:
X.head()

,day_of_week,dep_hour,DISTANCE,carrier_AA,carrier_AS,carrier_B6,carrier_DL,carrier_F9,carrier_G4,carrier_MQ,carrier_OH,carrier_OO,carrier_UA,carrier_WN,carrier_YX
0,0,5,569.0,1,0,0,0,0,0,0,0,0,0,0,0
1,0,7,569.0,1,0,0,0,0,0,0,0,0,0,0,0
2,0,8,569.0,1,0,0,0,0,0,0,0,0,0,0,0
3,0,10,569.0,1,0,0,0,0,0,0,0,0,0,0,0
4,0,16,569.0,1,0,0,0,0,0,0,0,0,0,0,0


In [77]:
Y = df_clean['ARR_DEL15'].astype(int)

In [78]:
train_mask = df_clean['FL_DATE'] < '2026-06-22'

In [79]:
X_train = X[train_mask]
Y_train = Y[train_mask]

In [80]:
X_test = X[~train_mask]
Y_test = Y[~train_mask]

In [81]:
X_train.shape


(414427, 15)

In [82]:
X_test.shape

(180768, 15)

In [83]:
Y_train.mean()


np.float64(0.2650382335127779)

In [84]:
Y_test.mean()

np.float64(0.23896928659939812)

In [85]:
lazy_pred = np.zeros(len(Y_test))

In [86]:
print(accuracy_score(Y_test, lazy_pred))

0.7610307134006019


In [87]:
print(recall_score(Y_test, lazy_pred))

0.0


In [88]:
model = LogisticRegression(max_iter=1000)

In [89]:
model.fit(X_train, Y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [90]:
pred = model.predict(X_test)

In [91]:
prob = model.predict_proba(X_test)[:, 1]

In [92]:
print(accuracy_score(Y_test, pred))
print(recall_score(Y_test, pred))
print(precision_score(Y_test, pred))
print(roc_auc_score(Y_test, prob))

0.7562068507700478
0.07053567294782166
0.4374102784955498
0.6659105849376242


In [93]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')

In [94]:
model.fit(X_train, Y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to

In [95]:
pred = model.predict(X_test)

In [96]:
prob = model.predict_proba(X_test)[:, 1]

In [97]:
print(accuracy_score(Y_test, pred))
print(recall_score(Y_test, pred))
print(precision_score(Y_test, pred))
print(roc_auc_score(Y_test, prob))

0.6346200654983183
0.6115792397796195
0.34904675712454913
0.6661794250043405


About a quarter of flights arrive 15 or more minutes late, so accuracy alone is misleading. A logistic regression using only information known before departure reached an AUC of 0.67, and weighting late flights raised recall from 7% to 61% at the cost of precision. I excluded departure delay on purpose because it is only known after the plane leaves, and using it would have been data leakage.